# Project 01 (basic) — The explore-exploit dilemma: multi-armed bandits

> **Module 13 — Reinforcement Learning** · Format: **Jupyter notebook**
>
> **Why this format?** Bandits are the simplest RL case (one state) and isolate the *one*
> question that distinguishes RL from everything else: **explore or exploit?** A notebook is
> ideal, because here the code, short experiments and above all the **learning-curve plots**
> belong together — you *see* the difference between the strategies directly.

## Goal
You build the **k-armed testbed** of Sutton & Barto and four action-selection strategies and
compare them empirically:
- **greedy** (always the best action so far — pure exploit),
- **ε-greedy** (with probability ε random),
- **optimistic initialization** (high initial values force early exploration),
- **UCB** (upper confidence bound — "optimism under uncertainty").

## Prior knowledge
Script module 13, section **3.1** (explore-exploit, bandits). Python/NumPy basics.

## What should work in the end
Two plots like in Sutton & Barto's chapter 2: **mean reward** and **% optimal action** over
time, averaged over many random bandit problems — plus your conclusion on which strategy wins
when.

> **How to work:** try the places marked with `# TODO` **yourself** first. The complete
> reference solution is in `solution/bandits_solution.ipynb`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng_global = np.random.default_rng(0)
print("numpy", np.__version__)

## 1 · The environment: the k-armed Gaussian testbed

A bandit has $k$ arms. For each arm $a$ there is a **true value** $q_*(a)$, drawn from
$\mathcal N(0,1)$. If you pull arm $a$, you get a **noisy** reward $R \sim \mathcal N(q_*(a), 1)$.
The agent does **not** know $q_*$ and has to estimate it from the observed rewards. The
**optimal** action is $a^* = \arg\max_a q_*(a)$.

This class is fully given — it is only the *environment*, not the learning part.


In [ ]:
class GaussianBandit:
    # k-armed bandit: q*(a) ~ N(0,1), reward R ~ N(q*(a), 1).
    def __init__(self, k=10, seed=None):
        self.k = k
        self.rng = np.random.default_rng(seed)
        self.q_true = self.rng.normal(0.0, 1.0, size=k)   # true values q*(a)
        self.optimal_action = int(np.argmax(self.q_true))

    def step(self, a):
        # Pull arm a, return a noisy reward.
        return self.rng.normal(self.q_true[a], 1.0)

# a small demo: one bandit, 5 pulls on arm 0
demo = GaussianBandit(k=10, seed=42)
print("true values q*(a):", np.round(demo.q_true, 2))
print("optimal arm:", demo.optimal_action, " q* =", round(demo.q_true[demo.optimal_action], 2))
print("5 rewards from arm 0:", np.round([demo.step(0) for _ in range(5)], 2))

## 2 · The agent

The agent estimates a value $Q(a)$ for each arm and counts the pulls $N(a)$. After each pull it
updates incrementally (**sample average**):
$$N(a)\leftarrow N(a)+1,\qquad Q(a)\leftarrow Q(a)+\tfrac1{N(a)}\big(R - Q(a)\big).$$

The **selection** depends on the strategy:
- `greedy` / `epsilon`: with probability $\varepsilon$ a random arm, otherwise $\arg\max_a Q(a)$
  (greedy = ε=0).
- `ucb`: choose $\arg\max_a\big[Q(a) + c\sqrt{\ln t / N(a)}\big]$; an arm never pulled
  ($N(a)=0$) has an infinite bonus, so it is tried **first**.
- The **optimistic initialization** is not a selection rule of its own but only a high initial
  value `Q_init` with otherwise greedy selection.

**Your task:** fill in `select_action` and `update`.


In [ ]:
class BanditAgent:
    def __init__(self, k=10, strategy="epsilon", epsilon=0.1, c=2.0, Q_init=0.0):
        self.k = k
        self.strategy = strategy      # "greedy" | "epsilon" | "ucb"
        self.epsilon = epsilon
        self.c = c
        self.Q = np.full(k, float(Q_init))   # value estimates (optimistic if Q_init is high)
        self.N = np.zeros(k, dtype=int)       # number of pulls per arm
        self.t = 0                            # total time step
        self.rng = np.random.default_rng()

    def select_action(self):
        self.t += 1
        # TODO: implement the selection depending on self.strategy.
        #  - "epsilon": with prob. self.epsilon self.rng.integers(self.k),
        #               otherwise argmax(self.Q). ("greedy" = epsilon 0.)
        #  - "ucb": prefer arms with N==0 first (infinite bonus);
        #           otherwise argmax(self.Q + self.c*sqrt(ln(self.t)/self.N)).
        # Tip on ties: np.argmax takes the first -> for a fair random choice
        #   on ties you can use np.flatnonzero(x==x.max()) + rng.choice.
        raise NotImplementedError

    def update(self, a, r):
        # TODO: incremental sample-average update for arm a with reward r.
        raise NotImplementedError

## 3 · The experiment harness

A single bandit is too noisy to compare strategies. That is why we use the **testbed**: average
over **many** random bandit problems (`n_runs`), each over `n_steps` pulls. Per step we log (a)
the **reward** received and (b) whether the **optimal action** was chosen. Given — nothing to do
here.


In [ ]:
def run_experiment(agent_kwargs, k=10, n_steps=1000, n_runs=2000, seed=0):
    seeder = np.random.default_rng(seed)
    rewards = np.zeros((n_runs, n_steps))
    optimal = np.zeros((n_runs, n_steps))
    for run in range(n_runs):
        env = GaussianBandit(k=k, seed=int(seeder.integers(1 << 30)))
        agent = BanditAgent(k=k, **agent_kwargs)
        agent.rng = np.random.default_rng(int(seeder.integers(1 << 30)))
        for t in range(n_steps):
            a = agent.select_action()
            r = env.step(a)
            agent.update(a, r)
            rewards[run, t] = r
            optimal[run, t] = 1.0 if a == env.optimal_action else 0.0
    return rewards.mean(axis=0), optimal.mean(axis=0)

# quick check with few runs (the big experiment comes afterwards)
avg_r, avg_opt = run_experiment(dict(strategy="epsilon", epsilon=0.1), n_steps=200, n_runs=200)
print("after 200 steps:  mean reward =", round(avg_r[-1], 3),
      "| %optimal =", round(100*avg_opt[-1], 1))

## 4 · The big comparison

We compare five configurations (the classic Sutton & Barto curves):

| Label | strategy | parameter |
|---|---|---|
| greedy | `greedy` | — |
| ε=0.1 | `epsilon` | ε=0.1 |
| ε=0.01 | `epsilon` | ε=0.01 |
| optimistic, greedy | `greedy` | Q_init=5 |
| UCB c=2 | `ucb` | c=2 |

> The full experiment (2000 runs × 1000 steps × 5 configs) takes **~1–2 min** on the CPU. For a
> fast test you can reduce `N_RUNS`/`N_STEPS`.


In [ ]:
N_STEPS, N_RUNS = 1000, 2000   # set smaller if needed, e.g. 500/500

configs = {
    "greedy":               dict(strategy="greedy"),
    "$\\varepsilon$=0.1":   dict(strategy="epsilon", epsilon=0.1),
    "$\\varepsilon$=0.01":  dict(strategy="epsilon", epsilon=0.01),
    "optimistic, greedy":   dict(strategy="greedy", Q_init=5.0),
    "UCB c=2":              dict(strategy="ucb", c=2.0),
}

results = {}
for i, (label, kw) in enumerate(configs.items()):
    results[label] = run_experiment(kw, k=10, n_steps=N_STEPS, n_runs=N_RUNS, seed=100 + i)
    print(f"done: {label:22s}  %optimal(final) = {100*results[label][1][-1]:.1f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for label, (r, o) in results.items():
    ax1.plot(r, label=label, lw=1.3)
    ax2.plot(100 * o, label=label, lw=1.3)
ax1.set(xlabel="step", ylabel="mean reward", title="Mean reward")
ax2.set(xlabel="step", ylabel="% optimal action", title="% optimal action", ylim=(0, 100))
for ax in (ax1, ax2):
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5 · Regret

The **regret** is the summed *missed* reward relative to always-optimal play:
$$\rho_T = \sum_{t=1}^{T}\big(q_*(a^*) - q_*(A_t)\big).$$
A good strategy has **sublinear** (ideally logarithmic) regret — the curve flattens. We
approximate $q_*(a^*)-q_*(A_t)$ over the testbed by the mean reward shortfall to the theoretical
optimum (the expected value of the maximum of 10 standard normals ≈ 1.54).


In [ ]:
# E[max of 10 N(0,1)] ~ 1.5388 (Monte-Carlo estimate of the testbed optimum)
q_star_max = np.mean([GaussianBandit(k=10, seed=s).q_true.max() for s in range(5000)])
print("mean optimum q*(a*) ~", round(q_star_max, 3))

plt.figure(figsize=(7, 4.5))
for label, (r, o) in results.items():
    regret = np.cumsum(q_star_max - r)
    plt.plot(regret, label=label, lw=1.4)
plt.xlabel("step"); plt.ylabel("cumulative regret")
plt.title("Regret (lower = better; flat = sublinear)")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6 · Observations & conclusion

Write down your observations (answered in the solution). Typical for the 10-armed testbed:

- **greedy** rises quickly but gets stuck early (~ only around 30–40 % optimal action) — it
  fixates on an arm that happened to look good and never explores again. **Pure exploit loses.**
- **ε=0.1** explores strongly → finds the best arm quickly, but the constant exploration caps it
  at ~91 % optimal action. **ε=0.01** is slower but better in the long run (less "interference").
  → **The explore-exploit trade-off in one picture.**
- **optimistic initialization** (Q₀=5) explores *by itself* at the beginning (every unpulled arm
  looks too good) → a characteristic *hump* early, then very good — but only an **initial** trick
  (useless for non-stationary problems).
- **UCB** is usually the overall winner here: targeted exploration by uncertainty instead of blind
  randomness → fast **and** high, with the flattest regret.

**The core message:** there is no "right" ε — every strategy is a different point in the
explore-exploit trade-off. Exactly this dilemma returns in full RL (the next projects), only with
**many states**.

### Mini-tasks to try further
1. Set UCB's `c` to 0.5 and 4 — how does the curve change?
2. Build a **non-stationary** variant: let `q_true` drift slightly per step
   (`q_true += rng.normal(0, 0.01, k)`). Why does a **constant** step update
   $Q\leftarrow Q+\alpha(R-Q)$ now beat the sample average? (→ script 2.2: "forgets old
   experience").
3. Add a **softmax/Boltzmann** selection ($\propto e^{Q(a)/\tau}$) and compare.

> Write down your answers first, then compare them with the reference answers in the `solution/`
> notebook.
